# Neural Style Transfer (Local)

Apply the artistic style of a reference image to a content image using a pre-trained VGG19 network. Everything runs locally on your machine - no AWS / SageMaker required.

**References:**
- https://github.com/udacity/deep-learning-v2-pytorch/tree/master/style-transfer
- https://elix-tech.github.io/ja/2016/08/22/art.html


In [ ]:
# import resources
%matplotlib inline

from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import torch
from torchvision import transforms, models

# Make the style-transfer functions in scripts/train.py importable
sys.path.insert(0, os.path.join(os.getcwd(), 'scripts'))
from train import load_image, im_convert, get_features, gram_matrix, train


In [ ]:
# Directory used for input images and the result image
data_dir = 'data'
os.makedirs(data_dir, exist_ok=True)

# Use GPU when available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Using device:', device)


In [ ]:
# Names of the images placed in the data directory
input_image_name = 'mychild.jpg'
reference_image_name = 'illustration.jpg'

# Verify the image files exist before training
content_path = os.path.join(data_dir, input_image_name)
style_path = os.path.join(data_dir, reference_image_name)
print('Content image exists:', os.path.exists(content_path))
print('Style image exists:  ', os.path.exists(style_path))


In [ ]:
# Load in content and style image
content = load_image(content_path).to(device)
# Resize style to match content, makes code easier
style = load_image(style_path, shape=content.shape[-2:]).to(device)


In [ ]:
# Display the content and style images side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
ax1.imshow(im_convert(content))
ax1.set_title('Content Image')
ax2.imshow(im_convert(style))
ax2.set_title('Style Image')
plt.show()


## Run style transfer locally

The optimization runs directly in this notebook (no SageMaker training job, no S3). We use a frozen pre-trained VGG19 to extract features, then optimize the target image so that it matches the content image's structure (`conv4_2` feature map) and the style image's texture (Gram matrices of `conv1_1`–`conv5_1`).


In [ ]:
# Pre-trained VGG19 as a fixed feature extractor
vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1).features
for param in vgg.parameters():
    param.requires_grad_(False)
vgg.to(device)
vgg.eval()

content_features = get_features(content, vgg)
style_features = get_features(style, vgg)
style_grams = {layer: gram_matrix(style_features[layer]) for layer in style_features}

# Target image = copy of the content image, optimized via gradient descent
target = content.clone().requires_grad_(True).to(device)
optimizer = torch.optim.Adam([target], lr=0.003)


In [ ]:
# Run the style transfer optimization (tweak epochs as needed)
epochs = 500
train(vgg, target, content_features, style_features, style_grams,
      epochs, optimizer)


In [ ]:
# Save the stylized result locally
result_path = os.path.join(data_dir, 'result.jpg')
result = Image.fromarray(np.uint8(im_convert(target) * 255))
result.save(result_path)
print('Saved result image to', result_path)


In [ ]:
# Display the content image and the final stylized result
if os.path.exists(result_path):
    result_image = Image.open(result_path).convert('RGB')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
    ax1.imshow(im_convert(content))
    ax1.set_title('Content Image')
    ax2.imshow(result_image)
    ax2.set_title('Stylized Result')
    plt.show()
else:
    print('Result image not found. Run the training cell first.')
